# Modelagem - Credit Card Fraud Detection Dataset


# Preparação inicial dos dados para modelagem

## 1. Importando o dataset via Kaggle

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Path to dataset files: /kaggle/input/creditcardfraud


## 2. Garantindo integridade dos dados.
- Transformar `Time` para `Hour`
- Remover duplicatas

> **Nota sobre escalonamento:** o `RobustScaler` no `Amount` é aplicado **depois** da divisão treino/teste (seção 1 — Benchmarking). O scaler é ajustado (`fit`) apenas nos dados de treino e depois aplicado (`transform`) ao teste. Aplicar antes do split seria *data leakage*: a mediana e o IQR calculados sobre o dataset completo incluiriam informação do conjunto de teste, contaminando o escalonamento do treino.

In [ ]:
import pandas as pd
import os

# Integridade e Carregamento
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)
df = pd.read_csv(full_path)

# Remoção de duplicatas
duplicados = df.duplicated().sum()
print(f"\n--- Duplicatas: {duplicados} ---")
df = df.drop_duplicates() if duplicados > 0 else df
print(f"Formato após limpeza: {df.shape}")

# Converter segundos para horas (0-23)
df['Hour'] = (df['Time'] // 3600) % 24

# Remove Time (já extraímos Hour); Amount permanece na escala original —
# será escalonado com RobustScaler APÓS o split (seção 1), sem leakage.
df.drop('Time', axis=1, inplace=True)

print("Colunas atuais no DataFrame:", df.columns.tolist())
display(df.head())

# Próximos Passos e Estratégia de Modelagem

Para as próximas etapas, seguiremos o seguinte roteiro para garantir a máxima performance na detecção de fraudes:

1.  **Divisão Estratificada**: Realizar o split entre treino e teste (80/20) garantindo que a proporção de fraudes seja mantida em ambos os sets.
2.  **Cost-Sensitive Learning**: Em vez de balancear o dataset (o que pode remover informações valiosas das transações legítimas), utilizaremos o parâmetro `scale_pos_weight` no **XGBoost**. Isso forçará o modelo a penalizar severamente erros cometidos na classe minoritária (Fraude).
3.  **Otimização com Optuna**: Utilizar busca de hiperparâmetros focada exclusivamente na métrica **PR-AUC** (Area Under the Precision-Recall Curve), que é mais robusta para datasets desbalanceados do que a acurácia ou a curva ROC.
4.  **Calibração de Probabilidades**: Implementar **Platt Scaling** ou **Isotonic Regression** para calibrar as saídas do modelo, garantindo que as probabilidades estimadas reflitam o risco real para a operação de negócio.

## 1. Benchmarking: Comparação de Modelos

Para escolher o modelo que melhor se adeque ao nosso dataset e ao problema que queremos resolver, comparamos a performance de três diferentes modelos:

1. **Regressão Logística (Baseline):** Um modelo linear simples usado como ponto de partida para medir a complexidade do problema.
2. **Random Forest (Ensemble):** Um conjunto de árvores de decisão que utiliza `class_weight='balanced'` para tentar compensar a raridade das fraudes.
3. **XGBoost (Gradient Boosting):** Modelo de alto desempenho que utiliza **Cost-Sensitive Learning** através do parâmetro `scale_pos_weight`. Diferente de outros métodos, ele penaliza erros na classe minoritária proporcionalmente ao desbalanceamento, permitindo alta revocação sem a necessidade de reamostragem (oversampling/undersampling).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, average_precision_score

# 1. Separação features / target (Amount ainda na escala original)
X = df.drop('Class', axis=1)
y = df['Class']

# 2. Divisão Estratificada 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Escalonamento do Amount — fit APENAS no treino para evitar data leakage
rs = RobustScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train['scaled_amount'] = rs.fit_transform(X_train[['Amount']])
X_test['scaled_amount']  = rs.transform(X_test[['Amount']])
X_train.drop('Amount', axis=1, inplace=True)
X_test.drop('Amount',  axis=1, inplace=True)

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Fraudes no treino: {y_train.sum()} | Fraudes no teste: {y_test.sum()}")

# 4. Cálculo do scale_pos_weight para o XGBoost
count_legit = (y_train == 0).sum()
count_fraud = (y_train == 1).sum()
spw = count_legit / count_fraud

# --- MODELO 1: Regressão Logística ---
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log  = log_reg.predict(X_test)
y_probs_log = log_reg.predict_proba(X_test)[:, 1]
auccpr_log  = average_precision_score(y_test, y_probs_log)

# --- MODELO 2: Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf  = rf_model.predict(X_test)
y_probs_rf = rf_model.predict_proba(X_test)[:, 1]
auccpr_rf  = average_precision_score(y_test, y_probs_rf)

# --- MODELO 3: XGBoost (Cost-Sensitive) ---
xgb_model = XGBClassifier(
    scale_pos_weight=spw,
    learning_rate=0.1,
    n_estimators=100,
    max_depth=6,
    random_state=42,
    eval_metric='aucpr'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb  = xgb_model.predict(X_test)
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]
auccpr_xgb  = average_precision_score(y_test, y_probs_xgb)

# --- RESUMO COMPARATIVO ---
print("\n--- Relatório: Regressão Logística ---")
print(classification_report(y_test, y_pred_log))

print("\n--- Relatório: Random Forest ---")
print(classification_report(y_test, y_pred_rf))

print("\n--- Relatório: XGBoost ---")
print(classification_report(y_test, y_pred_xgb))

print("=== PERFORMANCE COMPARATIVA (PR-AUC) ===")
print(f"Regressão Logística: {auccpr_log:.4f}")
print(f"Random Forest:       {auccpr_rf:.4f}")
print(f"XGBoost:             {auccpr_xgb:.4f} (Vencedor)")

### 1.1. Justificativa da Escolha do Modelo (XGBoost)

A escolha do **XGBoost** (PR-AUC: 0.8103) como modelo final, superando a Regressão Logística (0.6936) e o Random Forest (0.8047), baseia-se nos seguintes pontos:

*   **Superioridade em PR-AUC:** Em problemas de detecção de fraude, a métrica **PR-AUC (Area Under the Precision-Recall Curve)** é muito mais relevante que a Acurácia ou o F1-Score isolado. Como o dataset é extremamente desbalanceado (99.8% legítimas), a acurácia é enganosa. O PR-AUC foca especificamente na performance da classe minoritária (Fraude), avaliando o quão bem o modelo consegue separar as classes em diferentes limiares de decisão.
*   **Equilíbrio entre Precisão e Revocação (Recall):** Embora o Random Forest tenha apresentado uma Precisão ligeiramente maior (0.97 vs 0.83), o XGBoost entregou uma **Revocação significativamente melhor (0.79 vs 0.69)**. Para o negócio, é preferível investigar alguns falsos positivos extras do que deixar passar 10% a mais de fraudes reais.
*   **Por que não F1-Score?** O F1-Score assume um peso igual entre Precisão e Recall. No entanto, o PR-AUC nos dá uma visão holística da capacidade de ranking do modelo, sendo mais estável para otimização quando lidamos com proporções tão pequenas de eventos positivos.
*   **Robustez ao Desbalanceamento via Cost-Sensitive Learning:** Através do parâmetro `scale_pos_weight`, o XGBoost foi capaz de aprender as características complexas das fraudes sem a necessidade de gerar dados artificiais (SMOTE), capturando as não-linearidades que identificamos previamente na visualização t-SNE.

## 2. Comparação: Cost-Sensitive Learning vs. SMOTE

O README deste projeto afirma que **Cost-Sensitive Learning é preferível ao SMOTE** para este problema. Até aqui, essa era uma escolha metodológica justificada teoricamente mas nunca testada empiricamente dentro do projeto. Esta seção corrige isso.

**O que será comparado:**
- **Cost-Sensitive (CS):** XGBoost com `scale_pos_weight` — penaliza erros na classe minoritária sem alterar os dados
- **SMOTE:** XGBoost treinado sobre dados com oversampling sintético da classe minoritária — sem `scale_pos_weight`

**Protocolo correto para evitar leakage:**
O SMOTE é aplicado **somente no conjunto de treino de cada fold**, após o split. Aplicá-lo antes do split contaminaria o teste com amostras sintéticas derivadas de transações reais do teste — um erro comum que infla artificialmente as métricas.

A comparação é repetida em **5 seeds** para verificar se o resultado é consistente e não depende de uma divisão específica.

In [ ]:
!pip install imbalanced-learn -q

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

results_cs, results_sm = [], []

for seed in range(5):
    Xtr_raw, Xte_raw, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

    # Scaling correto: fit apenas no treino de cada seed
    rs_cmp = RobustScaler()
    Xtr = Xtr_raw.copy()
    Xte = Xte_raw.copy()
    Xtr['scaled_amount'] = rs_cmp.fit_transform(Xtr[['Amount']])
    Xte['scaled_amount']  = rs_cmp.transform(Xte[['Amount']])
    Xtr.drop('Amount', axis=1, inplace=True)
    Xte.drop('Amount', axis=1, inplace=True)

    spw_r = (ytr == 0).sum() / (ytr == 1).sum()

    # --- Cost-Sensitive ---
    cs_model = XGBClassifier(
        scale_pos_weight=spw_r, learning_rate=0.1,
        n_estimators=100, max_depth=6,
        eval_metric='aucpr', random_state=42
    )
    cs_model.fit(Xtr, ytr)
    ap_cs = average_precision_score(yte, cs_model.predict_proba(Xte)[:, 1])

    # --- SMOTE (aplicado apenas no treino, após o split) ---
    smote = SMOTE(random_state=seed)
    Xtr_sm, ytr_sm = smote.fit_resample(Xtr, ytr)

    sm_model = XGBClassifier(
        learning_rate=0.1, n_estimators=100,
        max_depth=6, eval_metric='aucpr', random_state=42
    )
    sm_model.fit(Xtr_sm, ytr_sm)
    ap_sm = average_precision_score(yte, sm_model.predict_proba(Xte)[:, 1])

    results_cs.append(ap_cs)
    results_sm.append(ap_sm)
    print(f"seed {seed}: Cost-Sensitive={ap_cs:.4f} | SMOTE={ap_sm:.4f} | diff(CS-SM)={ap_cs - ap_sm:+.4f}")

m_cs, s_cs = np.mean(results_cs), np.std(results_cs)
m_sm, s_sm = np.mean(results_sm), np.std(results_sm)
diff_mean   = m_cs - m_sm

print(f"\nCost-Sensitive — Média: {m_cs:.4f} ± {s_cs:.4f}")
print(f"SMOTE          — Média: {m_sm:.4f} ± {s_sm:.4f}")
print(f"Diferença média (CS − SMOTE): {diff_mean:+.4f}")

if abs(diff_mean) < max(s_cs, s_sm):
    veredito = "DENTRO DO RUÍDO — as abordagens são equivalentes para este dataset"
elif diff_mean > 0:
    veredito = "Cost-Sensitive supera SMOTE de forma consistente"
else:
    veredito = "SMOTE supera Cost-Sensitive de forma consistente"
print(f"Veredito: {veredito}")

### 2.1 Análise Crítica: Cost-Sensitive vs. SMOTE

#### Resultado observado

```
Cost-Sensitive: ~0.811 ± 0.034
SMOTE:          ~0.804 ± 0.035
Diferença (CS − SMOTE): ~+0.007
```

A diferença de +0.007 em PR-AUC é **menos de um quinto do desvio-padrão (~0.034)**. Usando a mesma régua da seção 3.2 (teste de robustez): o ganho é **dentro do ruído** — as duas abordagens são estatisticamente equivalentes para este dataset.

Isso se confirma pela alternância de vencedores entre seeds: Cost-Sensitive ganha em 3 das 5 seeds, SMOTE ganha em 2. Não há dominância consistente.

#### Como interpretar

A originalidade do projeto — usar Cost-Sensitive em vez de SMOTE — está **empiricamente sustentada como "não inferior"**, mesmo que não prove superioridade estrita. Para um problema com ~95 fraudes no teste, a variância é alta demais para detectar diferenças pequenas entre abordagens igualmente competentes.

A **preferência por Cost-Sensitive continua justificada por razões práticas**, independentemente do empate em PR-AUC:

| Critério | Cost-Sensitive | SMOTE |
|----------|---------------|-------|
| Dados sintéticos | Não gera | Gera (~94.000 amostras extras) |
| Risco de leakage se mal aplicado | Baixo | Alto (SMOTE pré-split é erro comum) |
| Complexidade do pipeline | Baixa (1 parâmetro) | Maior (1 etapa extra) |
| Preserva distribuição original | Sim | Não |
| Performance neste dataset | ~0.811 | ~0.804 |

#### O que esta comparação muda no projeto

A afirmação do README deixa de ser uma preferência metodológica e passa a ser um **resultado empírico documentado**: "testamos as duas abordagens em 5 divisões independentes e verificamos que são equivalentes em PR-AUC, com leve vantagem para Cost-Sensitive — o que valida nossa escolha pelas razões adicionais listadas acima."

# 3. Otimização de Hiperparâmetros (Optuna)

Busca Bayesiana pelos hiperparâmetros que maximizam o PR-AUC, usando o conjunto de treino **sem duplicatas** definido no benchmark (seção 1).

> **Protocolo sem vazamento + validação cruzada:** cada conjunto de hiperparâmetros é avaliado por **validação cruzada estratificada** (`StratifiedKFold`) *dentro do conjunto de treino*. Isso (a) mantém o **teste intocado** durante toda a busca e (b) evita o overfitting a um único conjunto de validação pequeno — o score de cada *trial* é a **média do PR-AUC entre os folds**, uma estimativa mais estável. O teste é usado uma só vez, na avaliação final (seção 3.1).

A otimização focará em:
1.  **Complexidade da Árvore**: `max_depth` e `min_child_weight`.
2.  **Robustez**: `subsample` e `colsample_bytree` para evitar overfitting.
3.  **Regularização**: `gamma`, `alpha` (L1) e `lambda` (L2).

### 3.1. Treinamento e avaliação do modelo otimizado

Aplicamos os melhores hiperparâmetros encontrados pela busca para treinar o modelo otimizado no **treino completo** e avaliá-lo, **uma única vez**, no conjunto de teste. O resultado é comparado ao **XGBoost do benchmark** (sem otimização) — a decisão sobre qual usar como detector final é discutida na seção 3.3.

In [6]:
# Modelo final com os melhores hiperparâmetros (selecionados por validação cruzada).
# Reajuste no treino COMPLETO para aproveitar todos os dados de treino.
best_params = dict(study.best_params)   # cópia, para não mutar o estudo
best_params['scale_pos_weight'] = spw   # peso do treino (benchmark, sem duplicatas)
best_params['eval_metric'] = 'aucpr'

final_xgb = XGBClassifier(**best_params, random_state=42)
final_xgb.fit(X_train, y_train)

# Avaliação Final: o conjunto de TESTE é usado aqui pela primeira e única vez (sem vazamento)
y_probs_final = final_xgb.predict_proba(X_test)[:, 1]
auccpr_final = average_precision_score(y_test, y_probs_final)

print(f"PR-AUC médio na CV (seleção dos hiperparâmetros): {study.best_value:.4f}")
print(f"PR-AUC do XGBoost benchmark (sem otimização):     {auccpr_xgb:.4f}")
print(f"PR-AUC Final no TESTE (modelo otimizado):         {auccpr_final:.4f}")
print(f"Ganho sobre o benchmark:                          {auccpr_final - auccpr_xgb:.4f}")

PR-AUC médio na CV (seleção dos hiperparâmetros): 0.8491
PR-AUC do XGBoost benchmark (sem otimização):     0.8103
PR-AUC Final no TESTE (modelo otimizado):         0.8099
Ganho sobre o benchmark:                          -0.0004


### 3.2. Teste de robustez: o ganho é real ou ruído?

O resultado de um único split é frágil: com ~95 fraudes no teste, diferenças de PR-AUC da ordem de ±0.01 podem ser puro acaso da divisão. Para checar, reavaliamos o modelo **otimizado** contra **dois baselines**, em **5 divisões diferentes** (variando a *seed*, com split estratificado):

- **XGBoost default** — parâmetros padrão da biblioteca + `scale_pos_weight`;
- **XGBoost benchmark** — a configuração manual da seção 1 (`lr=0.1, n_est=100, max_depth=6`).

Comparar com os dois é importante porque *"ganhar do benchmark"* e *"ganhar de um bom default"* são perguntas diferentes. Regra de leitura: se o **ganho médio for menor que o desvio-padrão**, ele é **indistinguível de zero** (ruído).

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# Teste de robustez: o ganho do modelo otimizado é real ou ruído?
# Comparamos o modelo OTIMIZADO contra DOIS baselines, em 5 divisões treino/teste
# (split estratificado, variando a seed). Para cada seed, o RobustScaler é ajustado
# apenas no treino daquele split — sem leakage entre splits.
#   - DEFAULT:   XGBoost com parâmetros padrão + scale_pos_weight
#   - BENCHMARK: a config manual da seção 1 (lr=0.1, n_est=100, max_depth=6)

opt_params = {k: v for k, v in study.best_params.items()
              if k not in ('scale_pos_weight', 'eval_metric', 'random_state')}

g_default, g_bench = [], []
for seed in range(5):
    Xtr_raw, Xte_raw, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

    # Scaling correto: fit apenas no treino de cada seed
    rs_fold = RobustScaler()
    Xtr = Xtr_raw.copy()
    Xte = Xte_raw.copy()
    Xtr['scaled_amount'] = rs_fold.fit_transform(Xtr[['Amount']])
    Xte['scaled_amount']  = rs_fold.transform(Xte[['Amount']])
    Xtr.drop('Amount', axis=1, inplace=True)
    Xte.drop('Amount',  axis=1, inplace=True)

    spw_r = (ytr == 0).sum() / (ytr == 1).sum()

    def pr_auc(model):
        model.fit(Xtr, ytr)
        return average_precision_score(yte, model.predict_proba(Xte)[:, 1])

    ap_default = pr_auc(XGBClassifier(scale_pos_weight=spw_r, eval_metric='aucpr', random_state=42))
    ap_bench   = pr_auc(XGBClassifier(scale_pos_weight=spw_r, learning_rate=0.1, n_estimators=100,
                                      max_depth=6, eval_metric='aucpr', random_state=42))
    ap_opt     = pr_auc(XGBClassifier(**opt_params, scale_pos_weight=spw_r,
                                      eval_metric='aucpr', random_state=42))

    g_default.append(ap_opt - ap_default)
    g_bench.append(ap_opt - ap_bench)
    print(f"seed {seed}: otim={ap_opt:.4f} | otim-default={ap_opt-ap_default:+.4f} | otim-benchmark={ap_opt-ap_bench:+.4f}")

def veredito(diffs, nome):
    m, s = np.mean(diffs), np.std(diffs)
    tag = "DENTRO DO RUÍDO" if abs(m) < s else ("GANHO consistente" if m > 0 else "PIORA consistente")
    print(f"Ganho otim vs {nome}: {m:+.4f} ± {s:.4f}  ->  {tag}")

print()
veredito(g_default, "DEFAULT  ")
veredito(g_bench,   "BENCHMARK")

### 3.3. Análise Crítica: Vale a pena usar Optuna?

#### Resultado observado

| Modelo | PR-AUC (seed=42) | PR-AUC médio (5 seeds) |
|--------|-----------------|------------------------|
| XGBoost benchmark (lr=0.1, max_depth=6) | 0.8103 | ~0.811 |
| XGBoost otimizado (Optuna, 20 trials) | 0.8099 | ~0.811 |
| Ganho | −0.0004 | ~0.000 |

O modelo otimizado pelo Optuna **não supera o benchmark de forma consistente**. O ganho médio entre as 5 seeds é inferior ao desvio-padrão — dentro do ruído estatístico para este tamanho de teste (~95 fraudes).

#### Por que isso acontece?

1. **Ceiling effect do dataset:** os features V1–V28 já são componentes principais (PCA aplicado pelo banco), e o dataset tem apenas 492 fraudes. Com tão poucos exemplos positivos, a separabilidade das classes é alta e o benchmark simples já captura boa parte do sinal disponível.

2. **Sobreajuste da CV interna:** o PR-AUC médio na validação cruzada foi 0.849 — muito acima do 0.810 no teste. Isso indica que o Optuna encontrou configurações que se ajustam bem aos folds de treino mas não generalizam melhor do que um benchmark simples.

3. **Poucas fraudes no teste:** com ~95 fraudes no conjunto de teste, diferenças de PR-AUC < 0.01 são estatisticamente inestimáveis — podem ser puro acaso de quais fraudes caíram no teste.

#### Conclusão operacional

Para **este dataset**, o Optuna não produz ganho mensurável. Isso **não invalida a abordagem** — em datasets maiores ou com menos sinal nos features brutos, a busca bayesiana é essencial. O valor da seção é demonstrar o protocolo correto (CV dentro do treino, teste intocado durante a busca) — que é aplicável a qualquer problema futuro.

## 4. Calibração de Probabilidades

O XGBoost com `scale_pos_weight` aprende a **discriminar** fraudes de legítimas — mas os scores que ele retorna não são necessariamente **probabilidades bem calibradas**.

**O problema concreto:** o parâmetro `scale_pos_weight ≈ 600` força o modelo a tratar cada fraude como se valesse 600 vezes mais do que uma transação legítima. Como efeito colateral, o modelo se torna extremamente agressivo com a classe positiva: mais de 50% das fraudes recebem score exatamente igual a 1.000. Isso significa que o sistema de risco não consegue **diferenciar** uma transação com 80% de chance de fraude de uma com 99% — ambas chegam com score ≈ 1.0.

**O que a calibração faz:** aplica uma transformação pós-treinamento que mapeia os scores brutos do modelo para probabilidades que reflitam a taxa real de fraude observada. Duas técnicas serão comparadas:

- **Platt Scaling:** ajusta uma regressão logística sobre os scores do modelo. Assume relação sigmoidal entre score e probabilidade — simples e eficaz quando a distorção é monotônica.
- **Isotonic Regression:** regressão não-paramétrica — mais flexível, aprende qualquer forma de distorção, mas requer mais dados de calibração para ser estável.

**Protocolo:** usamos `CalibratedClassifierCV` com `cv=3`, que treina o classificador em 3 folds e calibra cada fold nos dados restantes — sem tocar o conjunto de teste.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless backend compatible with Colab
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss
from xgboost import XGBClassifier

# Base model (same config as benchmark)
base_xgb = XGBClassifier(
    scale_pos_weight=spw, learning_rate=0.1,
    n_estimators=100, max_depth=6,
    eval_metric='aucpr', random_state=42
)

# Platt Scaling
platt_xgb = CalibratedClassifierCV(
    XGBClassifier(scale_pos_weight=spw, learning_rate=0.1,
                  n_estimators=100, max_depth=6,
                  eval_metric='aucpr', random_state=42),
    method='sigmoid', cv=3
)

# Isotonic Regression
iso_xgb = CalibratedClassifierCV(
    XGBClassifier(scale_pos_weight=spw, learning_rate=0.1,
                  n_estimators=100, max_depth=6,
                  eval_metric='aucpr', random_state=42),
    method='isotonic', cv=3
)

base_xgb.fit(X_train, y_train)
platt_xgb.fit(X_train, y_train)
iso_xgb.fit(X_train, y_train)

p_base  = base_xgb.predict_proba(X_test)[:, 1]
p_platt = platt_xgb.predict_proba(X_test)[:, 1]
p_iso   = iso_xgb.predict_proba(X_test)[:, 1]

models = {
    'Sem calibracao': p_base,
    'Platt Scaling':  p_platt,
    'Isotonic':       p_iso,
}

print(f'{"Modelo":<20} {"PR-AUC":>8} {"Brier":>10} {"Alertas@0.5":>13} {"Prec@0.5":>10} {"MaxScore(fraude)":>18}')
print('-' * 82)
for name, probs in models.items():
    prauc  = average_precision_score(y_test, probs)
    brier  = brier_score_loss(y_test, probs)
    alerts = int((probs >= 0.5).sum())
    tp     = int(((probs >= 0.5) & (y_test == 1)).sum())
    prec   = tp / alerts if alerts > 0 else 0
    max_fr = float(probs[y_test == 1].max())
    print(f'{name:<20} {prauc:>8.4f} {brier:>10.6f} {alerts:>13d} {prec:>10.3f} {max_fr:>18.4f}')

# Reliability diagram
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], 'k--', label='Perfeitamente calibrado')
colors = ['tab:blue', 'tab:orange', 'tab:green']
for (name, probs), color in zip(models.items(), colors):
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', color=color, label=name)
ax.set_xlabel('Score medio previsto')
ax.set_ylabel('Fracao de positivos reais')
ax.set_title('Diagrama de Confiabilidade (Reliability Diagram)')
ax.legend()
plt.tight_layout()
plt.savefig('calibration_diagram.png', dpi=120)
plt.show()
print('Diagrama salvo como calibration_diagram.png')

### 4.1 Análise Crítica: A calibração melhora o modelo operacionalmente?

#### Resultados obtidos

| Método | PR-AUC | Brier Score | Alertas (limiar=0.5) | Precisão@0.5 | Score máx (fraude) |
|--------|--------|-------------|----------------------|--------------|--------------------|
| Sem calibração | 0.8199 | 0.000451 | 84 | 0.905 | 1.000 |
| Platt Scaling | 0.8151 | 0.000429 | 74 | 0.973 | 0.918 |
| Isotonic Regression | 0.8055 | 0.000436 | 71 | 0.986 | 1.000 |

#### Interpretação

**O problema antes da calibração:** o `scale_pos_weight ≈ 600` torna o modelo extremamente agressivo. Mais de 50% das fraudes recebem score exatamente igual a **1.000** — o sistema de risco não consegue distinguir uma transação com 70% de chance de fraude de uma com 99%. Todo o topo da distribuição está colapsado em 1.0.

**O que o Platt Scaling corrige:** comprime os scores extremos, distribuindo-os num intervalo mais amplo (máximo: 0.918 vs 1.000). Com isso:
- O número de alertas no limiar 0.5 cai de **84 → 74** (−12% de trabalho da equipe antifraude)
- A precisão sobe de **0.905 → 0.973** (+7.5%) — 73 dos 74 alertas são fraudes reais
- O Brier Score melhora ligeiramente (0.000451 → 0.000429), confirmando melhor calibração
- O custo: PR-AUC cai de 0.8199 → 0.8151 (−0.005) — perda pequena de poder discriminativo

**Isotonic Regression:** melhora ainda mais a precisão (0.986) mas mantém scores em 1.000 para as ~28 fraudes mais óbvias. Isso sugere overfitting da calibração nos dados de treino com poucos exemplos positivos.

#### Recomendação

**Platt Scaling é a escolha preferível** para este cenário:

| Critério | Sem calibração | Platt Scaling | Isotonic |
|----------|---------------|---------------|----------|
| Scores colapsados em 1.0 | Sim (>50% fraudes) | Não (max=0.918) | Parcial |
| Alertas por dia (limiar=0.5) | 84 | **74** | 71 |
| Precisão dos alertas | 0.905 | **0.973** | 0.986 |
| Estabilidade com poucos positivos | Alta | Alta | **Baixa** |
| PR-AUC | **0.820** | 0.815 | 0.806 |

Em termos operacionais: com Platt Scaling, um analista de fraude recebe **10 alertas a menos por dia** e **cada alerta investigado tem 97% de chance de ser fraude real** — uma melhoria de eficiência concreta sem sacrifício significativo de cobertura.